# 2026 FIFA World Cup Prediction

This notebook predicts the 2026 FIFA World Cup using:

- Transfermarkt national team profile data
- FotMob match statistics converted into pre-match recent 5-match form
- Historical international matches from the last year before the prediction date

Prediction date used here:

```text
as_of_date = 2026-06-11
```

That means World Cup fixtures on or after 2026-06-11 are treated as future matches. Recent form is calculated only from matches before this date.

Main outputs:

- Group-stage win/draw/loss probabilities
- Predicted scores
- Predicted group tables
- Qualified Round of 32 teams
- Simplified knockout bracket predictions
- Model-by-model comparison


## 1. Setup

Run this cell first. It imports packages, sets paths, and defines the prediction cutoff date.


In [23]:
import warnings
warnings.filterwarnings("ignore")

import re
import unicodedata
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, log_loss, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestRegressor,
)
from sklearn.neural_network import MLPClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False
    print("XGBoost is not available. To use it, run: pip install xgboost")

REQUIRED_FILES = [
    "fotmob_match_ids_extracted.csv",
    "fotmob_statistics_long.csv",
    "transfermarkt_country_profile.csv",
]


def find_data_dir():
    candidates = [
        Path.cwd() / "data" / "processed",
        Path.cwd() / "kaggle_dataset" / "input",
        Path("/kaggle/input"),
    ]

    for candidate in candidates:
        if candidate.exists() and all((candidate / f).exists() for f in REQUIRED_FILES):
            return candidate

    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        for candidate in kaggle_root.rglob("*"):
            if candidate.is_dir() and all((candidate / f).exists() for f in REQUIRED_FILES):
                return candidate

    raise FileNotFoundError(
        "Could not find required dataset files. "
        "Attach the Kaggle dataset, then check that it contains input/fotmob_match_ids_extracted.csv, "
        "input/fotmob_statistics_long.csv, and input/transfermarkt_country_profile.csv."
    )


DATA_DIR = find_data_dir()
PROJECT_ROOT = DATA_DIR.parents[1] if DATA_DIR.name == "input" else DATA_DIR.parent

MATCHES_PATH = DATA_DIR / "fotmob_match_ids_extracted.csv"
STATS_PATH = DATA_DIR / "fotmob_statistics_long.csv"
TRANSFERMARKT_PATH = DATA_DIR / "transfermarkt_country_profile.csv"

AS_OF_DATE = pd.Timestamp("2026-06-11")
TRAIN_START_DATE = AS_OF_DATE - pd.DateOffset(years=1)

print("Data folder:", DATA_DIR)
print("Matches path exists:", MATCHES_PATH.exists())
print("Stats path exists:", STATS_PATH.exists())
print("Transfermarkt path exists:", TRANSFERMARKT_PATH.exists())
print("Prediction date:", AS_OF_DATE.date())
print("Training period starts:", TRAIN_START_DATE.date())
print("XGBoost available:", HAS_XGBOOST)



Project root: /Users/minseobeom/Desktop/WorldCup2026
Data folder: /Users/minseobeom/Desktop/WorldCup2026/data/processed
Prediction date: 2026-06-11
Training period starts: 2025-06-11
XGBoost available: True


## 2. Helper Functions

These helpers normalize team names, parse numeric stat values, and build common prediction tables.


In [24]:
def normalize_team_name(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value)).strip()
    return re.sub(r"\s+", " ", text)


def to_number(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).replace(",", "")
    match = re.search(r"-?\d+(\.\d+)?", text)
    return float(match.group(0)) if match else np.nan


def make_target(row):
    if row["home_score"] > row["away_score"]:
        return "home_win"
    if row["home_score"] < row["away_score"]:
        return "away_win"
    return "draw"


## 3. Load Historical Match Results

This file contains international matches involving the 48 World Cup teams, plus FotMob match IDs.


In [25]:
matches = pd.read_csv(MATCHES_PATH)

matches["date"] = pd.to_datetime(matches["date"])
matches["home_team"] = matches["home_team"].apply(normalize_team_name)
matches["away_team"] = matches["away_team"].apply(normalize_team_name)
matches["fotmob_match_id"] = pd.to_numeric(matches["fotmob_match_id"], errors="coerce")

matches = matches.dropna(subset=["fotmob_match_id", "home_score", "away_score"]).copy()
matches["fotmob_match_id"] = matches["fotmob_match_id"].astype(int)
matches = matches.sort_values(["date", "fotmob_match_id"]).reset_index(drop=True)

print("Matches:", matches.shape)
print(matches["date"].min(), "~", matches["date"].max())
matches.head()


Matches: (1514, 26)
2023-01-06 00:00:00 ~ 2026-06-10 00:00:00


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,...,fotmob_home_long,fotmob_home_score,fotmob_away_team,fotmob_away_long,fotmob_away_score,fotmob_status,fotmob_utc_time,match_score,match_reversed,fotmob_match_status
0,2023-01-06,Iraq,Oman,0,0,Gulf Cup,Basra,Iraq,False,draw,...,Iraq,0.0,Oman,Oman,0.0,FT,2023-01-06T16:00:00.000Z,1.08,False,matched
1,2023-01-06,Yemen,Saudi Arabia,0,2,Gulf Cup,Basra,Iraq,True,away_win,...,Yemen,0.0,Saudi Arabia,Saudi Arabia,2.0,FT,2023-01-06T18:45:00.000Z,1.08,False,matched
2,2023-01-07,Kuwait,Qatar,0,2,Gulf Cup,Basra,Iraq,True,away_win,...,Kuwait,0.0,Qatar,Qatar,2.0,FT,2023-01-07T16:15:00.000Z,1.08,False,matched
3,2023-01-09,Iraq,Saudi Arabia,2,0,Gulf Cup,Basra,Iraq,False,home_win,...,Saudi Arabia,0.0,Iraq,Iraq,2.0,FT,2023-01-09T16:15:00.000Z,1.08,True,matched
4,2023-01-09,Sweden,Finland,2,0,Friendly,Faro-Loulé,Portugal,True,home_win,...,Sweden,2.0,Finland,Finland,0.0,FT,2023-01-09T18:45:00.000Z,1.08,False,matched


## 4. Load FotMob Statistics

Only a compact set of team-level stats is used. These are later converted into recent 5-match averages.


In [26]:
USE_STATS = [
    "BallPossesion",
    "total_shots",
    "ShotsOnTarget",
    "expected_goals",
    "accurate_passes",
    "corners",
    "yellow_cards",
    "red_cards",
    "fouls",
    "keeper_saves",
    "interceptions",
    "clearances",
    "duel_won",
    "touches_opp_box",
]

stats = pd.read_csv(STATS_PATH)
stats = stats[(stats["period"] == "All") & (stats["stat_key"].isin(USE_STATS))].copy()
stats["home_value_num"] = stats["home_value"].apply(to_number)
stats["away_value_num"] = stats["away_value"].apply(to_number)

home_stats_wide = stats.pivot_table(
    index="fotmob_match_id",
    columns="stat_key",
    values="home_value_num",
    aggfunc="first",
).add_prefix("stat_").reset_index()

away_stats_wide = stats.pivot_table(
    index="fotmob_match_id",
    columns="stat_key",
    values="away_value_num",
    aggfunc="first",
).add_prefix("stat_").reset_index()

print("Stats rows:", stats.shape)
print("Wide home stats:", home_stats_wide.shape)
home_stats_wide.head()


Stats rows: (25354, 13)
Wide home stats: (1336, 15)


stat_key,fotmob_match_id,stat_BallPossesion,stat_ShotsOnTarget,stat_accurate_passes,stat_clearances,stat_corners,stat_duel_won,stat_expected_goals,stat_fouls,stat_interceptions,stat_keeper_saves,stat_red_cards,stat_total_shots,stat_touches_opp_box,stat_yellow_cards
0,3859879,32.0,3.0,184.0,19.0,4.0,32.0,NaN,11.0,14.0,5.0,0.0,7.0,7.0,2.0
1,3859880,58.0,4.0,415.0,4.0,13.0,54.0,NaN,16.0,9.0,0.0,0.0,16.0,33.0,0.0
2,3859893,65.0,7.0,577.0,13.0,4.0,49.0,NaN,15.0,6.0,0.0,0.0,16.0,51.0,1.0
3,3859898,42.0,3.0,277.0,17.0,2.0,53.0,NaN,22.0,4.0,4.0,0.0,10.0,15.0,1.0
4,3859899,69.0,4.0,501.0,14.0,9.0,48.0,NaN,10.0,10.0,3.0,0.0,25.0,38.0,1.0


## 5. Build Team-Level Historical Rows

Each match becomes two rows: one from the home team's point of view and one from the away team's point of view.


In [27]:
home_rows = matches.copy()
home_rows["team"] = home_rows["home_team"]
home_rows["opponent"] = home_rows["away_team"]
home_rows["is_home"] = 1
home_rows["goals_for"] = home_rows["home_score"]
home_rows["goals_against"] = home_rows["away_score"]
home_rows = home_rows.merge(home_stats_wide, on="fotmob_match_id", how="left")

away_rows = matches.copy()
away_rows["team"] = away_rows["away_team"]
away_rows["opponent"] = away_rows["home_team"]
away_rows["is_home"] = 0
away_rows["goals_for"] = away_rows["away_score"]
away_rows["goals_against"] = away_rows["home_score"]
away_rows = away_rows.merge(away_stats_wide, on="fotmob_match_id", how="left")

team_rows = pd.concat([home_rows, away_rows], ignore_index=True)
team_rows["goal_diff"] = team_rows["goals_for"] - team_rows["goals_against"]
team_rows["points"] = np.select(
    [team_rows["goal_diff"] > 0, team_rows["goal_diff"] == 0],
    [3, 1],
    default=0,
)
team_rows = team_rows.sort_values(["team", "date", "fotmob_match_id"]).reset_index(drop=True)

print("Team rows:", team_rows.shape)
team_rows.head()


Team rows: (3028, 47)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,...,stat_expected_goals,stat_fouls,stat_interceptions,stat_keeper_saves,stat_red_cards,stat_total_shots,stat_touches_opp_box,stat_yellow_cards,goal_diff,points
0,2023-11-16,Qatar,Afghanistan,8,1,FIFA World Cup qualification,Al Rayyan,Qatar,False,home_win,...,NaN,15.0,8.0,9.0,1.0,2.0,2.0,3.0,-7,0
1,2024-06-06,Afghanistan,Qatar,0,0,FIFA World Cup qualification,Al Hofuf,Saudi Arabia,True,draw,...,NaN,15.0,6.0,2.0,0.0,2.0,10.0,4.0,0,1
2,2023-09-07,Czech Republic,Albania,1,1,UEFA Euro qualification,Prague,Czech Republic,False,draw,...,0.02,9.0,13.0,3.0,0.0,1.0,NaN,2.0,0,1
3,2023-10-12,Albania,Czech Republic,3,0,UEFA Euro qualification,Tirana,Albania,False,home_win,...,1.26,10.0,15.0,7.0,0.0,8.0,12.0,1.0,3,3
4,2024-03-25,Sweden,Albania,1,0,Friendly,Stockholm,Sweden,False,home_win,...,NaN,8.0,12.0,5.0,0.0,8.0,18.0,2.0,-1,0


## 6. Create Recent 5-Match Features

The `shift(1)` is important. It prevents the current match from being included in its own pre-match features.


In [28]:
base_rolling_cols = ["goals_for", "goals_against", "goal_diff", "points"]
stat_cols = [col for col in team_rows.columns if col.startswith("stat_")]
rolling_cols = base_rolling_cols + stat_cols

team_rows_with_form = []
for team, group in team_rows.groupby("team"):
    group = group.sort_values(["date", "fotmob_match_id"]).copy()
    previous_games = group[rolling_cols].shift(1)
    recent5 = previous_games.rolling(window=5, min_periods=1).mean()
    recent5.columns = [f"recent5_{col}" for col in recent5.columns]
    team_rows_with_form.append(pd.concat([group, recent5], axis=1))

team_rows = pd.concat(team_rows_with_form, ignore_index=True)
recent5_cols = [col for col in team_rows.columns if col.startswith("recent5_")]

print("Recent 5 feature count:", len(recent5_cols))
recent5_cols[:10]


Recent 5 feature count: 18


['recent5_goals_for',
 'recent5_goals_against',
 'recent5_goal_diff',
 'recent5_points',
 'recent5_stat_BallPossesion',
 'recent5_stat_ShotsOnTarget',
 'recent5_stat_accurate_passes',
 'recent5_stat_clearances',
 'recent5_stat_corners',
 'recent5_stat_duel_won']

## 7. Build Match-Level Training Dataset

Recent-form features are joined back to the original match rows for both home and away teams.


In [29]:
home_recent5 = team_rows[team_rows["is_home"] == 1][["fotmob_match_id", "team"] + recent5_cols].copy()
home_recent5 = home_recent5.rename(columns={"team": "home_team"})
home_recent5 = home_recent5.rename(columns={col: f"home_{col}" for col in recent5_cols})

away_recent5 = team_rows[team_rows["is_home"] == 0][["fotmob_match_id", "team"] + recent5_cols].copy()
away_recent5 = away_recent5.rename(columns={"team": "away_team"})
away_recent5 = away_recent5.rename(columns={col: f"away_{col}" for col in recent5_cols})

df = matches.merge(home_recent5, on=["fotmob_match_id", "home_team"], how="left")
df = df.merge(away_recent5, on=["fotmob_match_id", "away_team"], how="left")

print("Match-level rows:", df.shape)
df.head()


Match-level rows: (1514, 62)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,...,away_recent5_stat_corners,away_recent5_stat_duel_won,away_recent5_stat_expected_goals,away_recent5_stat_fouls,away_recent5_stat_interceptions,away_recent5_stat_keeper_saves,away_recent5_stat_red_cards,away_recent5_stat_total_shots,away_recent5_stat_touches_opp_box,away_recent5_stat_yellow_cards
0,2023-01-06,Iraq,Oman,0,0,Gulf Cup,Basra,Iraq,False,draw,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-06,Yemen,Saudi Arabia,0,2,Gulf Cup,Basra,Iraq,True,away_win,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-07,Kuwait,Qatar,0,2,Gulf Cup,Basra,Iraq,True,away_win,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-09,Iraq,Saudi Arabia,2,0,Gulf Cup,Basra,Iraq,False,home_win,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-01-09,Sweden,Finland,2,0,Friendly,Faro-Loulé,Portugal,True,home_win,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8. Add Transfermarkt Team Strength Features

Transfermarkt features are current team-strength indicators. For historical validation, this is a limitation because date-specific historical market values are not used.


In [30]:
tm = pd.read_csv(TRANSFERMARKT_PATH)
tm["country"] = tm["country"].apply(normalize_team_name)
tm = tm[["country", "squad_size", "average_age", "fifa_world_ranking", "total_market_value_eur"]].copy()

for col in ["squad_size", "average_age", "fifa_world_ranking", "total_market_value_eur"]:
    tm[col] = pd.to_numeric(tm[col], errors="coerce")

home_tm = tm.rename(columns={
    "country": "home_team",
    "squad_size": "home_squad_size",
    "average_age": "home_average_age",
    "fifa_world_ranking": "home_fifa_world_ranking",
    "total_market_value_eur": "home_total_market_value_eur",
})

away_tm = tm.rename(columns={
    "country": "away_team",
    "squad_size": "away_squad_size",
    "average_age": "away_average_age",
    "fifa_world_ranking": "away_fifa_world_ranking",
    "total_market_value_eur": "away_total_market_value_eur",
})

df = df.merge(home_tm, on="home_team", how="left")
df = df.merge(away_tm, on="away_team", how="left")

print("After Transfermarkt join:", df.shape)
df[["home_team", "away_team", "home_total_market_value_eur", "away_total_market_value_eur"]].head()


After Transfermarkt join: (1514, 70)


,home_team,away_team,home_total_market_value_eur,away_total_market_value_eur
0,Iraq,Oman,21200000.0,NaN
1,Yemen,Saudi Arabia,NaN,40680000.0
2,Kuwait,Qatar,NaN,19930000.0
3,Iraq,Saudi Arabia,21200000.0,40680000.0
4,Sweden,Finland,406080000.0,NaN


## 9. Final Feature Engineering and Training Window

The model is trained only on matches from the last year before the prediction date.


In [31]:
df["diff_squad_size"] = df["home_squad_size"] - df["away_squad_size"]
df["diff_average_age"] = df["home_average_age"] - df["away_average_age"]
df["diff_fifa_world_ranking"] = df["home_fifa_world_ranking"] - df["away_fifa_world_ranking"]
df["diff_total_market_value_eur"] = df["home_total_market_value_eur"] - df["away_total_market_value_eur"]

for col in recent5_cols:
    df[f"diff_{col}"] = df[f"home_{col}"] - df[f"away_{col}"]

df["target"] = df.apply(make_target, axis=1)

model_df = df[(df["date"] >= TRAIN_START_DATE) & (df["date"] < AS_OF_DATE)].copy()
model_df = model_df.sort_values("date").reset_index(drop=True)

print("Training rows:", len(model_df))
print("Training period:", model_df["date"].min(), "~", model_df["date"].max())
print(model_df["target"].value_counts())


Training rows: 439
Training period: 2025-06-14 00:00:00 ~ 2026-06-10 00:00:00
target
home_win    214
away_win    121
draw        104
Name: count, dtype: int64


## 10. Select Model Features


In [32]:
transfermarkt_features = [
    "home_squad_size",
    "away_squad_size",
    "home_average_age",
    "away_average_age",
    "home_fifa_world_ranking",
    "away_fifa_world_ranking",
    "home_total_market_value_eur",
    "away_total_market_value_eur",
    "diff_squad_size",
    "diff_average_age",
    "diff_fifa_world_ranking",
    "diff_total_market_value_eur",
]

recent5_features = []
for col in recent5_cols:
    recent5_features.extend([f"home_{col}", f"away_{col}", f"diff_{col}"])

feature_cols = transfermarkt_features + recent5_features + ["neutral"]
missing_features = [col for col in feature_cols if col not in model_df.columns]

print("Feature count:", len(feature_cols))
print("Missing features:", missing_features)


Feature count: 67
Missing features: []


## 11. Compare Classification Models

Models are evaluated with a time-based split: earlier 80% for training, latest 20% for validation.


In [33]:
model_df = model_df.sort_values("date").reset_index(drop=True)
split_idx = int(len(model_df) * 0.8)

train_df = model_df.iloc[:split_idx].copy()
valid_df = model_df.iloc[split_idx:].copy()

X_train = train_df[feature_cols]
y_train = train_df["target"]
X_valid = valid_df[feature_cols]
y_valid = valid_df["target"]

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_valid_enc = label_encoder.transform(y_valid)

candidate_models = {
    "LogisticRegression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
    ]),
    "RandomForest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=500, random_state=42, class_weight="balanced", min_samples_leaf=5, n_jobs=-1)),
    ]),
    "ExtraTrees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(n_estimators=500, random_state=42, class_weight="balanced", min_samples_leaf=5, n_jobs=-1)),
    ]),
    "GradientBoosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(random_state=42)),
    ]),
    "HistGradientBoosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(random_state=42, max_iter=300, learning_rate=0.05)),
    ]),
    "MLP_Dense_NeuralNet": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation="relu", solver="adam", alpha=0.001, learning_rate_init=0.001, max_iter=1000, early_stopping=True, random_state=42)),
    ]),
}

if HAS_XGBOOST:
    candidate_models["XGBoost"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, objective="multi:softprob", eval_metric="mlogloss", random_state=42, n_jobs=-1)),
    ])

model_results = []
validation_models = {}
for name, estimator in candidate_models.items():
    print("Training:", name)
    clf = clone(estimator)
    clf.fit(X_train, y_train_enc)
    pred_enc = clf.predict(X_valid)
    proba = clf.predict_proba(X_valid)
    model_results.append({
        "model": name,
        "accuracy": accuracy_score(y_valid_enc, pred_enc),
        "log_loss": log_loss(y_valid_enc, proba, labels=np.arange(len(label_encoder.classes_))),
    })
    validation_models[name] = clf

model_compare = pd.DataFrame(model_results).sort_values(["log_loss", "accuracy"], ascending=[True, False]).reset_index(drop=True)
model_compare


Training: LogisticRegression
Training: RandomForest
Training: ExtraTrees
Training: GradientBoosting
Training: HistGradientBoosting
Training: MLP_Dense_NeuralNet
Training: XGBoost


,model,accuracy,log_loss
0,RandomForest,0.602273,0.864791
1,GradientBoosting,0.602273,0.881228
2,MLP_Dense_NeuralNet,0.613636,0.884130
3,XGBoost,0.590909,0.886657
4,ExtraTrees,0.602273,0.890297
5,LogisticRegression,0.500000,1.173162
6,HistGradientBoosting,0.568182,1.786281


## 12. Train Final Models

All candidate classifiers are retrained on the full one-year training set. The best model by validation log loss is kept as `result_model`, but every trained model is also preserved for model-by-model World Cup predictions.


In [34]:
best_model_name = model_compare.iloc[0]["model"]
print("Best model by validation log loss:", best_model_name)

X_all = model_df[feature_cols]
y_all_enc = label_encoder.transform(model_df["target"])

trained_models = {}
for name, estimator in candidate_models.items():
    clf = clone(estimator)
    clf.fit(X_all, y_all_enc)
    trained_models[name] = clf

result_model = trained_models[best_model_name]

home_goal_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("regressor", RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=5)),
])

away_goal_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("regressor", RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=5)),
])

home_goal_model.fit(X_all, model_df["home_score"])
away_goal_model.fit(X_all, model_df["away_score"])

print("Final classifiers:", list(trained_models))
print("Score models trained.")


Best model by validation log loss: RandomForest
Final classifiers: ['LogisticRegression', 'RandomForest', 'ExtraTrees', 'GradientBoosting', 'HistGradientBoosting', 'MLP_Dense_NeuralNet', 'XGBoost']
Score models trained.


## 13. Build 2026 World Cup Group Fixtures

The groups below are editable. If the official draw changes, update this dictionary.

The notebook generates 6 round-robin matches per group, for 72 group-stage matches total.


In [35]:
group_teams = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Qatar", "Switzerland", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Colombia", "DR Congo", "Portugal", "Uzbekistan"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

host_teams = {"Mexico", "Canada", "United States"}
fixture_rows = []
match_id = 1

for group, teams in group_teams.items():
    for home_team, away_team in combinations(teams, 2):
        fixture_rows.append({
            "match_id": match_id,
            "date": AS_OF_DATE,
            "group": group,
            "home_team": normalize_team_name(home_team),
            "away_team": normalize_team_name(away_team),
            "neutral": normalize_team_name(home_team) not in host_teams,
        })
        match_id += 1

worldcup_fixtures = pd.DataFrame(fixture_rows)
worldcup_fixtures["date"] = pd.to_datetime(worldcup_fixtures["date"])

print("World Cup group fixtures:", len(worldcup_fixtures))
print(worldcup_fixtures.groupby("group").size())
worldcup_fixtures.head()


World Cup group fixtures: 72
group
A    6
B    6
C    6
D    6
E    6
F    6
G    6
H    6
I    6
J    6
K    6
L    6
dtype: int64


,match_id,date,group,home_team,away_team,neutral
0,1,2026-06-11,A,Mexico,South Africa,False
1,2,2026-06-11,A,Mexico,South Korea,False
2,3,2026-06-11,A,Mexico,Czech Republic,False
3,4,2026-06-11,A,South Africa,South Korea,True
4,5,2026-06-11,A,South Africa,Czech Republic,True


## 14. Build Pre-Match Features for World Cup Fixtures

The recent 5-match form is calculated only from matches before `AS_OF_DATE`.


In [36]:
history_team_rows = team_rows[team_rows["date"] < AS_OF_DATE].copy()
team_recent5 = (
    history_team_rows
    .sort_values(["team", "date", "fotmob_match_id"])
    .groupby("team")
    .tail(5)
)

recent5_summary = team_recent5.groupby("team")[rolling_cols].mean().reset_index()
recent5_summary = recent5_summary.rename(columns={col: f"recent5_{col}" for col in rolling_cols})
wc_recent5_cols = [col for col in recent5_summary.columns if col.startswith("recent5_")]

home_recent5_wc = recent5_summary.rename(columns={
    "team": "home_team",
    **{col: f"home_{col}" for col in wc_recent5_cols},
})
away_recent5_wc = recent5_summary.rename(columns={
    "team": "away_team",
    **{col: f"away_{col}" for col in wc_recent5_cols},
})

wc_pred_df = worldcup_fixtures.copy()
wc_pred_df = wc_pred_df.merge(home_recent5_wc, on="home_team", how="left")
wc_pred_df = wc_pred_df.merge(away_recent5_wc, on="away_team", how="left")
wc_pred_df = wc_pred_df.merge(home_tm, on="home_team", how="left")
wc_pred_df = wc_pred_df.merge(away_tm, on="away_team", how="left")

wc_pred_df["diff_squad_size"] = wc_pred_df["home_squad_size"] - wc_pred_df["away_squad_size"]
wc_pred_df["diff_average_age"] = wc_pred_df["home_average_age"] - wc_pred_df["away_average_age"]
wc_pred_df["diff_fifa_world_ranking"] = wc_pred_df["home_fifa_world_ranking"] - wc_pred_df["away_fifa_world_ranking"]
wc_pred_df["diff_total_market_value_eur"] = wc_pred_df["home_total_market_value_eur"] - wc_pred_df["away_total_market_value_eur"]

for col in wc_recent5_cols:
    wc_pred_df[f"diff_{col}"] = wc_pred_df[f"home_{col}"] - wc_pred_df[f"away_{col}"]

missing_wc_features = [col for col in feature_cols if col not in wc_pred_df.columns]
print("Missing World Cup features:", missing_wc_features)
print("World Cup prediction rows:", wc_pred_df.shape)
wc_pred_df.head()


Missing World Cup features: []
World Cup prediction rows: (72, 72)


,match_id,date,group,home_team,away_team,neutral,home_recent5_goals_for,home_recent5_goals_against,home_recent5_goal_diff,home_recent5_points,...,diff_recent5_stat_corners,diff_recent5_stat_duel_won,diff_recent5_stat_expected_goals,diff_recent5_stat_fouls,diff_recent5_stat_interceptions,diff_recent5_stat_keeper_saves,diff_recent5_stat_red_cards,diff_recent5_stat_total_shots,diff_recent5_stat_touches_opp_box,diff_recent5_stat_yellow_cards
0,1,2026-06-11,A,Mexico,South Africa,False,1.8,0.4,1.4,2.2,...,0.400000,-5.800000,NaN,1.4,1.600000,0.000000,0.0,-2.200000,-1.0,-0.600000
1,2,2026-06-11,A,Mexico,South Korea,False,1.8,0.4,1.4,2.2,...,-1.600000,2.400000,NaN,0.4,-3.800000,0.600000,0.0,-0.400000,-1.0,0.200000
2,3,2026-06-11,A,Mexico,Czech Republic,False,1.8,0.4,1.4,2.2,...,-3.733333,-27.733333,NaN,-5.4,-2.933333,-0.933333,0.0,-2.733333,-11.2,0.733333
3,4,2026-06-11,A,South Africa,South Korea,True,1.2,1.4,-0.2,1.0,...,-2.000000,8.200000,NaN,-1.0,-5.400000,0.600000,0.0,1.800000,0.0,0.800000
4,5,2026-06-11,A,South Africa,Czech Republic,True,1.2,1.4,-0.2,1.0,...,-4.133333,-21.933333,-0.52,-6.8,-4.533333,-0.933333,0.0,-0.533333,-10.2,1.333333


## 15. Predict Group Stage with the Best Model


In [37]:
 def align_score_with_result(home_score, away_score, predicted_result):
    home_score = int(home_score)
    away_score = int(away_score)

    if predicted_result == "home_win":
        if home_score <= away_score:
            home_score = away_score + 1

    elif predicted_result == "away_win":
        if away_score <= home_score:
            away_score = home_score + 1

    elif predicted_result == "draw":
        avg_score = int(round((home_score + away_score) / 2))
        home_score = avg_score
        away_score = avg_score

    return home_score, away_score


def predict_worldcup_group_stage(model_name, classifier):
    X_worldcup = wc_pred_df[feature_cols]

    pred_enc = classifier.predict(X_worldcup)
    pred_result = label_encoder.inverse_transform(pred_enc)
    pred_proba = classifier.predict_proba(X_worldcup)

    pred_home_goals = home_goal_model.predict(X_worldcup)
    pred_away_goals = away_goal_model.predict(X_worldcup)

    raw_home_scores = np.round(pred_home_goals).astype(int)
    raw_away_scores = np.round(pred_away_goals).astype(int)

    aligned_scores = [
        align_score_with_result(home_score, away_score, predicted_result)
        for home_score, away_score, predicted_result in zip(
            raw_home_scores,
            raw_away_scores,
            pred_result
        )
    ]

    result = wc_pred_df[[
        "match_id",
        "date",
        "group",
        "home_team",
        "away_team",
        "neutral"
    ]].copy()

    result["model"] = model_name
    result["predicted_result"] = pred_result

    for i, cls in enumerate(label_encoder.classes_):
        result[f"prob_{cls}"] = pred_proba[:, i]

    result["pred_home_goals"] = pred_home_goals
    result["pred_away_goals"] = pred_away_goals

    result["pred_home_score"] = [score[0] for score in aligned_scores]
    result["pred_away_score"] = [score[1] for score in aligned_scores]

    return result


wc_result = predict_worldcup_group_stage(best_model_name, result_model)

print("Predicted group-stage matches:", len(wc_result))

wc_result[[
    "match_id",
    "group",
    "home_team",
    "away_team",
    "predicted_result",
    "prob_home_win",
    "prob_draw",
    "prob_away_win",
    "pred_home_goals",
    "pred_away_goals",
    "pred_home_score",
    "pred_away_score"
]].head(10)

Predicted group-stage matches: 72


,match_id,group,home_team,away_team,predicted_result,prob_home_win,prob_draw,prob_away_win,pred_home_goals,pred_away_goals,pred_home_score,pred_away_score
0,1,A,Mexico,South Africa,home_win,0.439546,0.324978,0.235476,1.687090,0.927006,2,1
1,2,A,Mexico,South Korea,away_win,0.311382,0.320214,0.368404,1.128469,0.923777,1,2
2,3,A,Mexico,Czech Republic,draw,0.251137,0.418708,0.330155,1.217344,1.326014,1,1
3,4,A,South Africa,South Korea,away_win,0.235762,0.284800,0.479438,1.301163,1.567455,1,2
4,5,A,South Africa,Czech Republic,away_win,0.214932,0.364601,0.420467,1.067992,1.509761,1,2
5,6,A,South Korea,Czech Republic,away_win,0.255174,0.358910,0.385916,1.204652,1.332358,1,2
6,7,B,Canada,Qatar,home_win,0.497465,0.375596,0.126939,1.717722,0.551547,2,1
7,8,B,Canada,Switzerland,draw,0.295092,0.469929,0.234978,1.197881,0.753207,1,1
8,9,B,Canada,Bosnia and Herzegovina,home_win,0.467055,0.378113,0.154832,2.063146,0.684624,2,1
9,10,B,Qatar,Switzerland,away_win,0.200840,0.354659,0.444502,1.020517,1.645690,1,2


## 16. Group Table and Qualification Functions


In [38]:
def build_group_table(group_matches):
    rows = []
    for _, row in group_matches.iterrows():
        group = row["group"]
        home = row["home_team"]
        away = row["away_team"]
        hg = int(row["pred_home_score"])
        ag = int(row["pred_away_score"])

        home_pts = 3 if hg > ag else 1 if hg == ag else 0
        away_pts = 3 if ag > hg else 1 if ag == hg else 0

        rows.append({
            "group": group,
            "team": home,
            "played": 1,
            "wins": int(hg > ag),
            "draws": int(hg == ag),
            "losses": int(hg < ag),
            "goals_for": hg,
            "goals_against": ag,
            "goal_diff": hg - ag,
            "points": home_pts,
        })
        rows.append({
            "group": group,
            "team": away,
            "played": 1,
            "wins": int(ag > hg),
            "draws": int(ag == hg),
            "losses": int(ag < hg),
            "goals_for": ag,
            "goals_against": hg,
            "goal_diff": ag - hg,
            "points": away_pts,
        })

    table = pd.DataFrame(rows)
    table = table.groupby(["group", "team"], as_index=False).sum(numeric_only=True)
    table = table.sort_values(
        ["group", "points", "goal_diff", "goals_for", "wins"],
        ascending=[True, False, False, False, False],
    )
    table["group_rank"] = table.groupby("group").cumcount() + 1
    return table


def get_qualified_teams(group_table):
    top2 = group_table[group_table["group_rank"].isin([1, 2])].copy()
    thirds = group_table[group_table["group_rank"] == 3].copy()
    best_thirds = thirds.sort_values(
        ["points", "goal_diff", "goals_for", "wins"],
        ascending=[False, False, False, False],
    ).head(8).copy()

    qualified = pd.concat([top2, best_thirds], ignore_index=True)
    qualified = qualified.sort_values(
        ["group_rank", "points", "goal_diff", "goals_for", "wins"],
        ascending=[True, False, False, False, False],
    ).reset_index(drop=True)
    qualified["seed"] = np.arange(1, len(qualified) + 1)
    return qualified, best_thirds


def build_seeded_round32(qualified):
    seeded = qualified.sort_values("seed").reset_index(drop=True)
    pairs = []
    left = list(range(16))
    right = list(range(31, 15, -1))
    for match_no, (i, j) in enumerate(zip(left, right), start=1):
        pairs.append({
            "round": "R32",
            "match_no": match_no,
            "home_team": seeded.iloc[i]["team"],
            "away_team": seeded.iloc[j]["team"],
            "home_seed": int(seeded.iloc[i]["seed"]),
            "away_seed": int(seeded.iloc[j]["seed"]),
        })
    return pd.DataFrame(pairs)

group_table = build_group_table(wc_result)
qualified, best_thirds = get_qualified_teams(group_table)
round32 = build_seeded_round32(qualified)

print(group_table.groupby("group").size())
print("Qualified teams:", len(qualified))
round32.head()


group
A    4
B    4
C    4
D    4
E    4
F    4
G    4
H    4
I    4
J    4
K    4
L    4
dtype: int64
Qualified teams: 32


,round,match_no,home_team,away_team,home_seed,away_seed
0,R32,1,Belgium,Sweden,1,32
1,R32,2,Japan,Paraguay,2,31
2,R32,3,England,Bosnia and Herzegovina,3,30
3,R32,4,Turkey,Cape Verde,4,29
4,R32,5,Argentina,Mexico,5,28


## 17. Knockout Prediction Functions

This is a simplified seeded knockout bracket, not the official FIFA Round of 32 allocation table.


In [39]:
def make_fixture_features(home_team, away_team, neutral=True):
    fixture = pd.DataFrame([{
        "match_id": 0,
        "date": AS_OF_DATE,
        "home_team": normalize_team_name(home_team),
        "away_team": normalize_team_name(away_team),
        "neutral": neutral,
    }])

    fixture = fixture.merge(home_recent5_wc, on="home_team", how="left")
    fixture = fixture.merge(away_recent5_wc, on="away_team", how="left")
    fixture = fixture.merge(home_tm, on="home_team", how="left")
    fixture = fixture.merge(away_tm, on="away_team", how="left")

    fixture["diff_squad_size"] = fixture["home_squad_size"] - fixture["away_squad_size"]
    fixture["diff_average_age"] = fixture["home_average_age"] - fixture["away_average_age"]
    fixture["diff_fifa_world_ranking"] = fixture["home_fifa_world_ranking"] - fixture["away_fifa_world_ranking"]
    fixture["diff_total_market_value_eur"] = fixture["home_total_market_value_eur"] - fixture["away_total_market_value_eur"]

    for col in wc_recent5_cols:
        fixture[f"diff_{col}"] = fixture[f"home_{col}"] - fixture[f"away_{col}"]

    return fixture


def predict_knockout_match(classifier, home_team, away_team, round_name, match_no):
    fixture = make_fixture_features(home_team, away_team, neutral=True)
    X = fixture[feature_cols]

    pred_enc = classifier.predict(X)
    pred_label = label_encoder.inverse_transform(pred_enc)[0]
    proba = classifier.predict_proba(X)[0]
    prob_map = dict(zip(label_encoder.classes_, proba))

    pred_home_goals = home_goal_model.predict(X)[0]
    pred_away_goals = away_goal_model.predict(X)[0]
    home_score = int(round(pred_home_goals))
    away_score = int(round(pred_away_goals))

    if home_score > away_score:
        winner = home_team
    elif away_score > home_score:
        winner = away_team
    else:
        if prob_map.get("home_win", 0) >= prob_map.get("away_win", 0):
            winner = home_team
            home_score += 1
        else:
            winner = away_team
            away_score += 1

    return {
        "round": round_name,
        "match_no": match_no,
        "home_team": home_team,
        "away_team": away_team,
        "predicted_90min_result": pred_label,
        "prob_home_win": prob_map.get("home_win", np.nan),
        "prob_draw_90min": prob_map.get("draw", np.nan),
        "prob_away_win": prob_map.get("away_win", np.nan),
        "pred_home_goals": pred_home_goals,
        "pred_away_goals": pred_away_goals,
        "pred_home_score": home_score,
        "pred_away_score": away_score,
        "winner": winner,
    }


def predict_round(classifier, round_df, round_name):
    rows = []
    for _, row in round_df.iterrows():
        rows.append(predict_knockout_match(classifier, row["home_team"], row["away_team"], round_name, row["match_no"]))
    return pd.DataFrame(rows)


def build_next_round(previous_results, next_round_name):
    winners = previous_results.sort_values("match_no")["winner"].tolist()
    rows = []
    for match_no, i in enumerate(range(0, len(winners), 2), start=1):
        if i + 1 >= len(winners):
            break
        rows.append({
            "round": next_round_name,
            "match_no": match_no,
            "home_team": winners[i],
            "away_team": winners[i + 1],
        })
    return pd.DataFrame(rows)


def run_knockout_tournament(classifier, qualified):
    r32 = build_seeded_round32(qualified)
    r32_results = predict_round(classifier, r32, "R32")

    r16 = build_next_round(r32_results, "R16")
    r16_results = predict_round(classifier, r16, "R16")

    qf = build_next_round(r16_results, "QF")
    qf_results = predict_round(classifier, qf, "QF")

    sf = build_next_round(qf_results, "SF")
    sf_results = predict_round(classifier, sf, "SF")

    final = build_next_round(sf_results, "Final")
    final_results = predict_round(classifier, final, "Final")

    tournament_results = pd.concat([r32_results, r16_results, qf_results, sf_results, final_results], ignore_index=True)
    champion = final_results.iloc[0]["winner"]
    return tournament_results, champion


## 18. Run Knockout Tournament with the Best Model


In [40]:
tournament_results, champion = run_knockout_tournament(result_model, qualified)

print("Best model:", best_model_name)
print("Predicted champion:", champion)
tournament_results


Best model: RandomForest
Predicted champion: Germany


,round,match_no,home_team,away_team,predicted_90min_result,prob_home_win,prob_draw_90min,prob_away_win,pred_home_goals,pred_away_goals,pred_home_score,pred_away_score,winner
0,R32,1,Belgium,Sweden,home_win,0.576820,0.339739,0.083441,2.091507,0.554932,2,1,Belgium
1,R32,2,Japan,Paraguay,home_win,0.499009,0.300100,0.200891,1.651369,0.613372,2,1,Japan
2,R32,3,England,Bosnia and Herzegovina,home_win,0.552772,0.338524,0.108705,2.571475,0.767643,3,1,England
3,R32,4,Turkey,Cape Verde,home_win,0.621541,0.227665,0.150794,2.171456,0.697625,2,1,Turkey
4,R32,5,Argentina,Mexico,home_win,0.404203,0.360136,0.235661,1.936419,0.810357,2,1,Argentina
5,R32,6,Portugal,Ivory Coast,home_win,0.358842,0.345342,0.295816,1.704578,1.609399,3,2,Portugal
6,R32,7,France,Scotland,home_win,0.441215,0.279377,0.279408,1.617204,1.127187,2,1,France
7,R32,8,Morocco,Norway,draw,0.307107,0.443278,0.249615,1.806023,1.427350,2,1,Morocco
8,R32,9,Ecuador,Uruguay,home_win,0.503068,0.380902,0.116030,1.950344,0.402319,2,0,Ecuador
9,R32,10,Spain,Senegal,home_win,0.365453,0.308929,0.325618,1.894272,1.656082,3,2,Spain


## 19. Model-by-Model World Cup Comparison

This section keeps every trained classifier and compares group-stage predictions, qualified teams, knockout paths, and champions.


In [41]:
all_group_predictions = {}
all_group_tables = {}
all_qualified = {}
all_tournament_results = {}
model_champions = []

for model_name, classifier in trained_models.items():
    print("Running World Cup prediction for:", model_name)
    group_prediction = predict_worldcup_group_stage(model_name, classifier)
    group_table_model = build_group_table(group_prediction)
    qualified_model, best_thirds_model = get_qualified_teams(group_table_model)
    tournament_model, champion_model = run_knockout_tournament(classifier, qualified_model)

    all_group_predictions[model_name] = group_prediction
    all_group_tables[model_name] = group_table_model
    all_qualified[model_name] = qualified_model
    all_tournament_results[model_name] = tournament_model

    final_row = tournament_model[tournament_model["round"] == "Final"].iloc[0]
    model_champions.append({
        "model": model_name,
        "champion": champion_model,
        "final_home": final_row["home_team"],
        "final_away": final_row["away_team"],
        "final_score": f"{final_row['pred_home_score']}-{final_row['pred_away_score']}",
    })

model_champions = pd.DataFrame(model_champions)
model_champions


Running World Cup prediction for: LogisticRegression
Running World Cup prediction for: RandomForest
Running World Cup prediction for: ExtraTrees
Running World Cup prediction for: GradientBoosting
Running World Cup prediction for: HistGradientBoosting
Running World Cup prediction for: MLP_Dense_NeuralNet
Running World Cup prediction for: XGBoost


,model,champion,final_home,final_away,final_score
0,LogisticRegression,Argentina,Spain,Argentina,2-3
1,RandomForest,Germany,France,Germany,1-2
2,ExtraTrees,Brazil,Portugal,Brazil,2-3
3,GradientBoosting,Spain,England,Spain,1-2
4,HistGradientBoosting,Germany,Argentina,Germany,2-3
5,MLP_Dense_NeuralNet,Germany,Germany,Portugal,3-2
6,XGBoost,Spain,Argentina,Spain,2-3


## 20. Inspect Individual Model Outputs

Change `MODEL_TO_VIEW` to any model name in `trained_models`.


In [42]:
MODEL_TO_VIEW = best_model_name

print("Available models:", list(trained_models))
print("Viewing:", MODEL_TO_VIEW)

display(all_group_predictions[MODEL_TO_VIEW].head(20))
display(all_group_tables[MODEL_TO_VIEW])
display(all_qualified[MODEL_TO_VIEW])
display(all_tournament_results[MODEL_TO_VIEW])


Available models: ['LogisticRegression', 'RandomForest', 'ExtraTrees', 'GradientBoosting', 'HistGradientBoosting', 'MLP_Dense_NeuralNet', 'XGBoost']
Viewing: RandomForest


,match_id,date,group,home_team,away_team,neutral,model,predicted_result,prob_away_win,prob_draw,prob_home_win,pred_home_goals,pred_away_goals,pred_home_score,pred_away_score
0,1,2026-06-11,A,Mexico,South Africa,False,RandomForest,home_win,0.235476,0.324978,0.439546,1.687090,0.927006,2,1
1,2,2026-06-11,A,Mexico,South Korea,False,RandomForest,away_win,0.368404,0.320214,0.311382,1.128469,0.923777,1,2
2,3,2026-06-11,A,Mexico,Czech Republic,False,RandomForest,draw,0.330155,0.418708,0.251137,1.217344,1.326014,1,1
3,4,2026-06-11,A,South Africa,South Korea,True,RandomForest,away_win,0.479438,0.284800,0.235762,1.301163,1.567455,1,2
4,5,2026-06-11,A,South Africa,Czech Republic,True,RandomForest,away_win,0.420467,0.364601,0.214932,1.067992,1.509761,1,2
5,6,2026-06-11,A,South Korea,Czech Republic,True,RandomForest,away_win,0.385916,0.358910,0.255174,1.204652,1.332358,1,2
6,7,2026-06-11,B,Canada,Qatar,False,RandomForest,home_win,0.126939,0.375596,0.497465,1.717722,0.551547,2,1
7,8,2026-06-11,B,Canada,Switzerland,False,RandomForest,draw,0.234978,0.469929,0.295092,1.197881,0.753207,1,1
8,9,2026-06-11,B,Canada,Bosnia and Herzegovina,False,RandomForest,home_win,0.154832,0.378113,0.467055,2.063146,0.684624,2,1
9,10,2026-06-11,B,Qatar,Switzerland,True,RandomForest,away_win,0.444502,0.354659,0.200840,1.020517,1.645690,1,2


,group,team,played,wins,draws,losses,goals_for,goals_against,goal_diff,points,group_rank
0,A,Czech Republic,3,2,1,0,5,3,2,7,1
3,A,South Korea,3,2,0,1,5,4,1,6,2
1,A,Mexico,3,1,1,1,4,4,0,4,3
2,A,South Africa,3,0,0,3,3,6,-3,0,4
5,B,Canada,3,2,1,0,5,3,2,7,1
7,B,Switzerland,3,2,1,0,5,3,2,7,2
4,B,Bosnia and Herzegovina,3,1,0,2,4,5,-1,3,3
6,B,Qatar,3,0,0,3,3,6,-3,0,4
10,C,Morocco,3,2,1,0,6,4,2,7,1
8,C,Brazil,3,2,0,1,5,4,1,6,2


,group,team,played,wins,draws,losses,goals_for,goals_against,goal_diff,points,group_rank,seed
0,G,Belgium,3,3,0,0,7,1,6,9,1,1
1,F,Japan,3,3,0,0,6,2,4,9,1,2
2,L,England,3,3,0,0,6,2,4,9,1,3
3,D,Turkey,3,3,0,0,6,3,3,9,1,4
4,J,Argentina,3,3,0,0,6,3,3,9,1,5
5,K,Portugal,3,3,0,0,6,3,3,9,1,6
6,I,France,3,2,1,0,7,4,3,7,1,7
7,C,Morocco,3,2,1,0,6,4,2,7,1,8
8,E,Ecuador,3,2,1,0,6,4,2,7,1,9
9,H,Spain,3,2,1,0,6,4,2,7,1,10


,round,match_no,home_team,away_team,predicted_90min_result,prob_home_win,prob_draw_90min,prob_away_win,pred_home_goals,pred_away_goals,pred_home_score,pred_away_score,winner
0,R32,1,Belgium,Sweden,home_win,0.576820,0.339739,0.083441,2.091507,0.554932,2,1,Belgium
1,R32,2,Japan,Paraguay,home_win,0.499009,0.300100,0.200891,1.651369,0.613372,2,1,Japan
2,R32,3,England,Bosnia and Herzegovina,home_win,0.552772,0.338524,0.108705,2.571475,0.767643,3,1,England
3,R32,4,Turkey,Cape Verde,home_win,0.621541,0.227665,0.150794,2.171456,0.697625,2,1,Turkey
4,R32,5,Argentina,Mexico,home_win,0.404203,0.360136,0.235661,1.936419,0.810357,2,1,Argentina
5,R32,6,Portugal,Ivory Coast,home_win,0.358842,0.345342,0.295816,1.704578,1.609399,3,2,Portugal
6,R32,7,France,Scotland,home_win,0.441215,0.279377,0.279408,1.617204,1.127187,2,1,France
7,R32,8,Morocco,Norway,draw,0.307107,0.443278,0.249615,1.806023,1.427350,2,1,Morocco
8,R32,9,Ecuador,Uruguay,home_win,0.503068,0.380902,0.116030,1.950344,0.402319,2,0,Ecuador
9,R32,10,Spain,Senegal,home_win,0.365453,0.308929,0.325618,1.894272,1.656082,3,2,Spain


## 21. Save Outputs


In [44]:
OUTPUT_DIR = DATA_DIR / "model_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def safe_filename(name):
    return re.sub(r"[^A-Za-z0-9_]+", "_", str(name))


# 1. 최고 모델 결과 저장
best_safe_name = safe_filename(best_model_name)

wc_result.to_csv(
    OUTPUT_DIR / f"group_predictions_{best_safe_name}.csv",
    index=False,
    encoding="utf-8-sig"
)

group_table.to_csv(
    OUTPUT_DIR / f"group_table_{best_safe_name}.csv",
    index=False,
    encoding="utf-8-sig"
)

qualified.to_csv(
    OUTPUT_DIR / f"qualified_teams_{best_safe_name}.csv",
    index=False,
    encoding="utf-8-sig"
)

tournament_results.to_csv(
    OUTPUT_DIR / f"tournament_results_{best_safe_name}.csv",
    index=False,
    encoding="utf-8-sig"
)


# 2. 모델별 전체 결과 저장
for model_name in trained_models.keys():
    safe_name = safe_filename(model_name)

    if model_name in all_group_predictions:
        all_group_predictions[model_name].to_csv(
            OUTPUT_DIR / f"group_predictions_{safe_name}.csv",
            index=False,
            encoding="utf-8-sig"
        )

    if model_name in all_group_tables:
        all_group_tables[model_name].to_csv(
            OUTPUT_DIR / f"group_table_{safe_name}.csv",
            index=False,
            encoding="utf-8-sig"
        )

    if model_name in all_qualified:
        all_qualified[model_name].to_csv(
            OUTPUT_DIR / f"qualified_teams_{safe_name}.csv",
            index=False,
            encoding="utf-8-sig"
        )

    if model_name in all_tournament_results:
        all_tournament_results[model_name].to_csv(
            OUTPUT_DIR / f"tournament_results_{safe_name}.csv",
            index=False,
            encoding="utf-8-sig"
        )


# 3. 우승팀 비교 요약 저장
model_champions.to_csv(
    OUTPUT_DIR / "champions_by_model.csv",
    index=False,
    encoding="utf-8-sig"
)


# 4. 모델 검증 성능 비교표 저장
model_compare.to_csv(
    OUTPUT_DIR / "model_validation_scores.csv",
    index=False,
    encoding="utf-8-sig"
)


print("저장 완료:", OUTPUT_DIR)

print("저장된 파일 목록:")
for file in sorted(OUTPUT_DIR.glob("*.csv")):
    print(file.name)

저장 완료: /Users/minseobeom/Desktop/WorldCup2026/data/processed/model_outputs
저장된 파일 목록:
champions_by_model.csv
group_predictions_ExtraTrees.csv
group_predictions_GradientBoosting.csv
group_predictions_HistGradientBoosting.csv
group_predictions_LogisticRegression.csv
group_predictions_MLP_Dense_NeuralNet.csv
group_predictions_RandomForest.csv
group_predictions_XGBoost.csv
group_table_ExtraTrees.csv
group_table_GradientBoosting.csv
group_table_HistGradientBoosting.csv
group_table_LogisticRegression.csv
group_table_MLP_Dense_NeuralNet.csv
group_table_RandomForest.csv
group_table_XGBoost.csv
model_validation_scores.csv
qualified_teams_ExtraTrees.csv
qualified_teams_GradientBoosting.csv
qualified_teams_HistGradientBoosting.csv
qualified_teams_LogisticRegression.csv
qualified_teams_MLP_Dense_NeuralNet.csv
qualified_teams_RandomForest.csv
qualified_teams_XGBoost.csv
tournament_results_ExtraTrees.csv
tournament_results_GradientBoosting.csv
tournament_results_HistGradientBoosting.csv
tournament_r